In [ ]:
path_to_dataset = "" #FIXME: Add path to dataset here

In [ ]:
import os
import pandas as pd
from ultralytics import YOLO

**Create the needed dataset structure**

In [ ]:
df = pd.read_csv("../Training_data/yolo_annotations_dataset.csv")

# Filter only rows used for training/validation
df = df[df['used_in_training'] == True]

base_dir = "yolo_dataset"

for split in ['train', 'val']:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)

for image_filename, group in df.groupby('image_filename'):
    split = group['split'].iloc[0] # 'train' or 'val'
    if split not in ['train', 'val']:
        continue

    # 1. Symlink or copy images to the structured directory
    subfolder = image_filename[0].upper()
    src_img = os.path.join(path_to_dataset, subfolder, image_filename)
    dst_img = os.path.join(base_dir, "images", split, image_filename)

    if os.path.exists(src_img) and not os.path.exists(dst_img):
        # Use os.symlink on Linux/macOS to save space, or use shutil.copy for Windows compatibility
        try:
            os.symlink(os.path.abspath(src_img), dst_img)
        except OSError:
            import shutil
            shutil.copy(src_img, dst_img)

    # 2. Create the corresponding YOLO .txt label file
    label_filename = os.path.splitext(image_filename)[0] + '.txt'
    dst_label = os.path.join(base_dir, 'labels', split, label_filename)

    with open(dst_label, 'w') as f:
        for _, row in group.iterrows():
            # Format: class_id x_center y_center width height
            line = f"{row['class_id']} {row['x_center']} {row['y_center']} {row['width']} {row['height']}\n"
            f.write(line)

**Train the yolo model**

In [ ]:
# 1. Load a pre-trained YOLO26 Small model
model = YOLO('yolo26s.pt')

# 2. Train the model on yolo dataset
results = model.train(data="yolo_dataset.yaml", 
    # --- TRAINING LENGTH ---
    epochs=300,      
    patience=50,

    # --- RESOLUTION & MEMORY ---         
    imgsz=1024,          #1024 resolution forces YOLO to look closer
    batch=16,            
    device=0,            # Set to 'cpu' if training locally without a dedicated GPU
    mosaic=0.0,          
    scale=0.1,           
    single_cls=True,     
    box=10.0,             
    
    # --- SPEED & SCHEDULING ---
    workers=4,           # Adjust based on your CPU core count
    cache=False,         # Set to True only if you have enough system RAM
    cos_lr=True          
)